# Can the finance domain actually be backtested?

A first real run of `finance-scaled` for [Lightningfish](https://github.com/rajul-kk/LightningFish) on Kaggle's free T4 GPU, same Ollama/qwen2.5:7b setup and population size (24 agents, 4 rounds) as the other notebooks here.

Every other domain here has a real accuracy number, mostly negative, reported honestly. Finance never did: one code path was ladder-scored but had only 5 hand-picked events, nowhere near enough to say anything; the other could pull real events at scale but was broken, it imported a package that isn't installed. Both fixed: `pull_edgar_events()` now downloads real, point-in-time SEC 8-K filings through the same baseline-ladder machinery proven on coding and HN. This notebook is that machinery's first real run.

One more fix along the way: ground truth used to also fetch Reddit sentiment, searched relative to when the code ran rather than the filing date, so anything over a week old got unrelated chatter or nothing. Dropped entirely, the ladder only ever scored the price move, so nothing about the result changes, and this run needs no Reddit credentials, only a SEC EDGAR user agent.

Every other domain here has failed the ladder so far, so the honest prior is finance probably will too, still worth knowing for certain rather than leaving it "never actually tested".


---
## What the data is

Real 8-K filings pulled live from SEC EDGAR via `sec-edgar-downloader`, across 30 large- and mid-cap tickers spanning tech, finance, energy, retail, and a few high-volatility names (AMC, GME, RIVN) so the sample isn't just one sector's reporting calendar.

Each seed's context is the filing's own as-filed text, starting at its first "Item X.XX" section header. An earlier pass found 8-K filings open with thousands of characters of cover-page boilerplate (checkboxes, registrant address, a symbol table), and truncating from character 0 mostly captured that instead of the disclosure. Skipping to the real Item section fixed it, confirmed against real filings.

The ground truth is the stock's price move in the 72 hours after the filing date, from `yfinance`, which serves real historical bars by date. Filings newer than 3 days are excluded so that window is always fully settled by the time this scores anything.


---
## 1. Setup

Sidebar: **Accelerator → GPU**, **Internet → On**.

Install `zstd` before Ollama. Its installer needs it to extract, Kaggle's base image doesn't ship it, and skipping this step makes the install fail quietly, surfacing later as a confusing `FileNotFoundError: 'ollama'`.


In [ ]:
!apt-get -qq update > /dev/null 2>&1; apt-get -qq install -y zstd > /dev/null 2>&1
!zstd --version || echo "WARNING: zstd missing - the install below will fail"
!curl -fsSL https://ollama.com/install.sh | sh


In [ ]:
import shutil, subprocess, time, requests

if shutil.which("ollama") is None:
    raise RuntimeError(
        "ollama not found after install. Scroll up for the installer's error: "
        "usually zstd (cell above) or Internet disabled in the sidebar."
    )

subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
for _ in range(60):
    try:
        requests.get("http://localhost:11434/api/tags", timeout=2)
        print("ollama up"); break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("ollama installed but the server did not start")


In [ ]:
MODEL = "qwen2.5:7b"

!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!ollama pull {MODEL}

requests.post("http://localhost:11434/api/generate",
              json={"model": MODEL, "prompt": "hi", "stream": False, "keep_alive": -1},
              timeout=600)

for m in requests.get("http://localhost:11434/api/ps").json().get("models", []):
    vram = m.get("size_vram", 0) / 1e9
    print(f"{m['name']}: {vram:.2f} GB in VRAM")
    assert vram > 0, "model landed on CPU - enable the GPU accelerator, this is pointless otherwise"
print("GPU inference confirmed")


In [ ]:
!git clone --depth 1 https://github.com/rajul-kk/LightningFish.git /kaggle/working/lf
!pip -q install anthropic openai scipy requests pytest yfinance sec-edgar-downloader

import os, sys
os.chdir("/kaggle/working/lf")
sys.path.insert(0, "/kaggle/working/lf")

# Engine, HN, and finance suites: confirms pull_edgar_events, the boilerplate-
# skipping fix, and the Reddit-free ground truth are actually present in this
# clone, not just the CLI wiring.
!python -m pytest tests/core tests/hn tests/finance -q 2>&1 | tail -5


---
## 2. Configuration

SEC's fair access policy requires a real identifying user agent, your actual name and a working email, not an app name. Fill in the line below before running the next cell.


In [ ]:
import os

# REQUIRED: SEC EDGAR needs a real name and email, not a placeholder.
os.environ["SEC_EDGAR_USER_AGENT"] = "Your Name you@example.com"

N_AGENTS = 24    # matches the other Kaggle notebooks' validated GPU size
N_ROUNDS = 4
PULL_LIMIT = 30  # real 8-K filings to pull; expect close to this many to succeed

os.environ["LIGHTNINGFISH_MODEL"] = f"ollama:{MODEL}"
os.environ["LIGHTNINGFISH_N_AGENTS"] = str(N_AGENTS)
os.environ["LIGHTNINGFISH_N_ROUNDS"] = str(N_ROUNDS)
os.environ["LIGHTNINGFISH_LOCAL_TIMEOUT"] = "120"
os.environ["PYTHONUNBUFFERED"] = "1"

assert "@" in os.environ["SEC_EDGAR_USER_AGENT"] and "Your Name" not in os.environ["SEC_EDGAR_USER_AGENT"], (
    "Set a real 'Name email@domain.com' above before continuing."
)
print(f"{MODEL} | {N_AGENTS} agents x {N_ROUNDS} rounds | pulling up to {PULL_LIMIT} filings")


### Throughput check

Same shape as every other backtest event in this repo, ~26 model calls. Worth confirming the per-call cost before committing to 30 of them.


In [ ]:
import time
from lightningfish_core.llm_provider import make_provider

provider = make_provider(f"ollama:{MODEL}")
t0 = time.time()
for _ in range(3):
    provider.get_opinion("Output ONLY a number between -1 and 1.", "Rate: 0.5", f"ollama:{MODEL}")
per_call = (time.time() - t0) / 3
print(f"{per_call:.2f}s per call  ->  ~{per_call*26:.0f}s per event  "
      f"->  ~{per_call*26*PULL_LIMIT/60:.0f} min for {PULL_LIMIT} events")


---
## 3. Run it

The first pass just pulls and parses the filings, no simulation yet, so a bad `SEC_EDGAR_USER_AGENT` or a network hiccup fails fast instead of after an hour of GPU time.


In [ ]:
from lightningfish_finance.backtest_events import pull_edgar_events

t0 = time.time()
events = pull_edgar_events(n=PULL_LIMIT)
print(f"{len(events)} events pulled in {time.time()-t0:.0f}s")
for e in events[:5]:
    print(" ", e.event_id, "-", e.seed.event_type, "-", e.seed.raw_input["filing_text"][:80])
assert len(events) >= 15, "too few events pulled - check SEC_EDGAR_USER_AGENT and network access"


In [ ]:
!python -m tests.integration.run_backtest finance-scaled {PULL_LIMIT} 2>&1 | tee /kaggle/working/finance_scaled.log


### Reading the log

Same report shape as every other backtest here: `sim` accuracy against `majority`, `naive` (a content-free keyword baseline), and `single_llm` (one raw model call, no agents or rounds). `beats_baselines` has to PASS on every rung, and `p_value_vs_best` has to clear the significance bar, for this to be a result rather than noise, exactly the same standard the coding and HN domains were held to.

Whatever it says, it's the first honest number this domain has ever had. If it fails like the rest, that's not a disappointing outcome, it's the same finding closing the same gap the baseline ladder exists to close.


---
## 4. Save


In [ ]:
!cp -r .cache/lightningfish /kaggle/working/cache
!ls -la /kaggle/working/finance_scaled.log /kaggle/working/cache


The log and the pulled/simulated event cache are saved to `/kaggle/working/`, download from the notebook's Output tab. The cache means a later re-score against a different question costs nothing to redo.

Either way the result belongs in [METHODOLOGY.md](https://github.com/rajul-kk/LightningFish/blob/main/METHODOLOGY.md)'s "Worked results" table, next to the other domains.
